In [ ]:
from pathlib import Path
import pandas as pd
import re
from functools import reduce

# ---------------------------------------------------------------------------
BASE_DIR   = Path("base_station").resolve()   # run this notebook from the repository root
MODALITIES = {
    "GPS"  : "_out_gps",
    "LiDAR": "preprocessed_out_lidar",
    "RGB"  : "_out_rgb",
    "Radar": "_out_radar",          # keep your spelling here
    "RSS"  : "_out_cleaned_net",
}

# ---------------------------------------------------------------------------
def frame_id(modality: str, path: Path) -> str:
    """
    Return the common frame key used to join modalities.
    * For RSS, strip the trailing “…csv._rss_map_cleaned” part.
    * For everything else, grab the first run of digits in the file name.
    """
    name = path.stem               # without extension
    if modality == "RSS":
        # handles both “…csv._rss_map_cleaned” and “…_rss_map_cleaned”
        m = re.match(r"(\d+)(?:_?csv)?\._rss_map_cleaned$", name)
        if m:
            return m.group(1)      # e.g. "000123"
    # default: first digits block, else whole stem
    m = re.search(r"\d+", name)
    return m.group(0) if m else name


# ---------------------------------------------------------------------------
dfs = []
for col, suffix in MODALITIES.items():
    root = BASE_DIR / suffix
    if not root.exists():
        print(f"[warn] {root} does not exist – skipping")
        continue

    recs = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        episode = p.parent.name          # episode_1 …
        key     = frame_id(col, p)       # unified frame ID
        recs.append({"Episode": episode,
                     "Frame":   key,
                     col:       str(p)})
    dfs.append(pd.DataFrame(recs))

# outer‑join on Episode + Frame so every modality shares one row
def ojoin(l, r): return l.merge(r, on=["Episode", "Frame"], how="outer")
table = reduce(ojoin, dfs)
table = table.dropna(subset=MODALITIES.keys())        # <<< add this

# nice order & constant scene
table.insert(0, "Scene", BASE_DIR.name)          # "base_station"
table = table[["Scene", "Episode", "Frame", *MODALITIES.keys()]]

out_csv = BASE_DIR / "dataset_index_local.csv"
table.to_csv(out_csv, index=False)
print("saved →", out_csv)
